# 03_pc_<agreement>_<source>_to_<target> — deterministic pipeline enforcement

Use this run-all-safe notebook to enforce approved metadata controls, apply deterministic quality checks, and write valid/quarantine outputs. Metadata and lineage evidence are optional.

**You edit**
- Dataset/table names and source/target kinds
- Deterministic transformation logic cell
- Target write mode

**This notebook produces**
- Metadata-controlled valid output + quarantine output
- Compact handover summary
- Optional metadata/lineage evidence section


## Segment 1: Load shared config and runtime context

What this does: loads environment config and builds notebook runtime context.

Functions used: `setup_fabricops_notebook`, `load_fabric_config`, `validate_dq_rules`, `enforce_dq_rules`.

Output: runtime context for deterministic execution.


In [ ]:
%run 00_env_config


In [ ]:
import json
from pathlib import Path
from pyspark.sql import functions as F
from fabricops_kit.config import setup_fabricops_notebook, get_path, load_fabric_config
from fabricops_kit.fabric_input_output import lakehouse_table_read, warehouse_read
from fabricops_kit import enforce_dq_rules, assert_dq_passed, standardize_output_columns, profile_dataframe, check_schema_drift, check_partition_drift, check_profile_drift, summarize_drift_results, prepare_drift_baselines, classify_columns, build_governance_classification_records, write_governance_classifications, summarize_governance_classifications, build_lineage_records, build_lineage_handover_markdown, build_run_summary, render_run_summary_markdown
from fabricops_kit.fabric_input_output import lakehouse_table_write, warehouse_write


In [ ]:
USE_SAMPLE_DATA = True
ENV_NAME = "dev"
SOURCE_LAYER = "source"
TARGET_LAYER = "product"
SOURCE_KIND = "lakehouse"
TARGET_KIND = "lakehouse"
SOURCE_TABLE = "minimal_source" if USE_SAMPLE_DATA else "TODO_source_table"
TARGET_TABLE = "sample_agreement_output" if USE_SAMPLE_DATA else "TODO_target_table"
DQ_TABLE_NAME = TARGET_TABLE
DATASET_NAME = "sample_agreement_dataset" if USE_SAMPLE_DATA else "target_dataset"
WRITE_MODE = "overwrite"
RUN_OPTIONAL_ADVANCED_EVIDENCE = False


In [ ]:
CONFIG = load_fabric_config(CONFIG)
if not USE_SAMPLE_DATA:
    setup_fabricops_notebook(config=CONFIG, env=ENV_NAME, required_targets=["source", "unified", "product"])


## Schema fail-fast guard
Use an explicit required-column guard before transformation so failures are immediate and readable.


## Segment 2: Load source data and run schema fail-fast

What this does: reads source data from lakehouse/warehouse and applies a simple required-column guard.

Functions used: `get_path`, `lakehouse_table_read`, `warehouse_read`.

Output: source dataframe validated against required columns.


In [ ]:
# Optional: use this section when this workflow is needed.
# Example: load supplementary controls tables or feature flags used by your pipeline.


In [ ]:
if SOURCE_KIND == "lakehouse":
    source_path = get_path(ENV_NAME, SOURCE_LAYER, config=CONFIG)
    df_source = lakehouse_table_read(source_path, SOURCE_TABLE)
else:
    df_source = warehouse_read(env=ENV_NAME, target=SOURCE_LAYER, schema="dbo", table=SOURCE_TABLE, config=CONFIG)


In [ ]:
REQUIRED_SOURCE_COLUMNS = ["customer_id", "event_ts", "status", "amount"]
missing = sorted(set(REQUIRED_SOURCE_COLUMNS) - set(df_source.columns))
if missing:
    raise ValueError(f"Fail fast: missing required source columns: {missing}")


## Optional drift guardrails
Optional: use this section when this workflow is needed. Baselines are prepared explicitly and drift checks run only when enabled.


In [ ]:
source_profile_baseline = None
partition_baseline_snapshot = None
schema_baseline_snapshot = None

if RUN_OPTIONAL_ADVANCED_EVIDENCE:
    source_profile_current = profile_dataframe(df_source)
    drift_baselines = prepare_drift_baselines(
        current_profile=source_profile_current,
        baseline_profile=source_profile_baseline,
        baseline_schema_snapshot=schema_baseline_snapshot,
        baseline_partition_snapshot=partition_baseline_snapshot,
    )
    schema_drift_preview = check_schema_drift(
        df_source,
        dataset_name=DATASET_NAME,
        table_name=SOURCE_TABLE,
        baseline_snapshot=drift_baselines["schema"],
    )
    partition_drift_preview = check_partition_drift(
        df_source,
        dataset_name=DATASET_NAME,
        table_name=SOURCE_TABLE,
        partition_column="event_ts",
        business_keys=["customer_id"],
        baseline_snapshot=drift_baselines["partition"],
    )
    profile_drift_preview = check_profile_drift(
        source_profile_current,
        baseline_profile=drift_baselines["profile"],
    )
    drift_preview_summary = summarize_drift_results(
        schema_drift_result=schema_drift_preview,
        partition_drift_result=partition_drift_preview,
        profile_drift_result=profile_drift_preview,
    )
    display(drift_preview_summary)


In [ ]:
df_transformed = (
    df_source.withColumn("status", F.trim(F.lower(F.col("status"))))
    .withColumn("email", F.trim(F.lower(F.col("email"))))
    .withColumn("amount", F.col("amount").cast("double"))
)


## Segment 3: Load approved active DQ rules from metadata

What this does: loads deterministic, human-approved rules from metadata for pipeline enforcement.

Functions used: `enforce_dq_rules`.


In [ ]:
# Prepare standardized frame for DQ enforcement.


In [ ]:
# Optional: use this section when this workflow is needed.
# Example runtime standardization before DQ enforcement.
# df_standard = standardize_output_columns(df_transformed, run_id=RUN_ID)
df_standard = df_transformed


In [ ]:
metadata_dq_rules = spark.table("METADATA_DQ_RULES")
dq = enforce_dq_rules(
    df_standard,
    table_name=DQ_TABLE_NAME,
    metadata_df=metadata_dq_rules,
    dq_run_id=RUN_ID,
)


In [ ]:
display(dq.rule_results)
display(dq.valid_rows)
display(dq.quarantine_rows)
display(dq.failure_rows)

df_valid = dq.valid_rows
df_quarantine = dq.quarantine_rows
dq_failure_evidence = dq.failure_rows

assert_dq_passed(dq.rule_results)


In [ ]:
# Optional output metadata strictness checks can be added in project implementations.


## Segment 4: Run deterministic DQ, split outputs, and publish evidence

What this does: enforces approved rules, splits valid/quarantine rows, and records row-level failure evidence.

Functions used: `enforce_dq_rules`, `assert_dq_passed`.


In [ ]:
# Optional: use this section when this workflow is needed.
# Example output writes with real helpers.
if TARGET_KIND == "lakehouse":
    target_path = get_path(ENV_NAME, TARGET_LAYER, config=CONFIG)
    lakehouse_table_write(df_valid, target_path, TARGET_TABLE, mode=WRITE_MODE)
else:
    warehouse_write(df_valid, env=ENV_NAME, target=TARGET_LAYER, schema="dbo", table=TARGET_TABLE, mode=WRITE_MODE, config=CONFIG)


In [ ]:
df_written = df_valid if USE_SAMPLE_DATA else (lakehouse_table_read(get_path(ENV_NAME, TARGET_LAYER, config=CONFIG), TARGET_TABLE) if TARGET_KIND == "lakehouse" else warehouse_read(env=ENV_NAME, target=TARGET_LAYER, schema="dbo", table=TARGET_TABLE, config=CONFIG))


In [ ]:
print("Optional: profiling metadata evidence can be generated in the optional section below.")


In [ ]:
print("Optional: quality run records can be generated in the optional section below.")


## Lineage notes

Accepted lineage patterns:
- paste exported notebook source into a project implementation when deterministic scanning is needed
- keep the minimal template on manual lineage notes/handover summary


In [ ]:
print("Optional: lineage evidence can be generated in the optional section below.")


In [ ]:
# Optional: use this section when this workflow is needed.
if RUN_OPTIONAL_ADVANCED_EVIDENCE:
    source_profile = profile_dataframe(df_source)
    output_profile = profile_dataframe(df_valid)

    drift_baselines = prepare_drift_baselines(current_profile=source_profile)
    schema_drift_result = check_schema_drift(df_valid, dataset_name=DATASET_NAME, table_name=TARGET_TABLE, baseline_snapshot=drift_baselines["schema"])
    partition_drift_result = check_partition_drift(df_valid, dataset_name=DATASET_NAME, table_name=TARGET_TABLE, partition_column="event_ts", business_keys=["customer_id"], baseline_snapshot=drift_baselines["partition"])
    profile_drift_result = check_profile_drift(source_profile, baseline_profile=drift_baselines["profile"])
    drift_summary = summarize_drift_results(schema_drift_result=schema_drift_result, partition_drift_result=partition_drift_result, profile_drift_result=profile_drift_result)

    column_names = list(df_valid.columns)
    classifications = classify_columns(column_names)
    governance_records = build_governance_classification_records(DATASET_NAME, TARGET_TABLE, classifications, run_id=RUN_ID)
    # write_governance_classifications(governance_records, metadata_path=get_path(ENV_NAME, "metadata", config=CONFIG))
    governance_summary = summarize_governance_classifications(classifications)

    lineage_records = build_lineage_records(dataset_name=DATASET_NAME, run_id=RUN_ID, source_tables=[SOURCE_TABLE], target_table=TARGET_TABLE, transformation_steps=[{"step": 1, "source": SOURCE_TABLE, "target": TARGET_TABLE, "operation": "DQ-controlled valid/quarantine split"}])
    lineage_markdown = build_lineage_handover_markdown(lineage_records, dataset_name=DATASET_NAME, run_id=RUN_ID)

    run_summary = build_run_summary(runtime_context=runtime_context, source_profile=source_profile, output_profile=output_profile, schema_drift_result=schema_drift_result, incremental_safety_result=partition_drift_result, quality_result={"status": "passed", "rule_count": len(dq.rules)}, lineage_summary={"record_count": len(lineage_records), "markdown": lineage_markdown})
    run_summary_markdown = render_run_summary_markdown(run_summary)

    print(drift_summary)
    print(governance_summary)
    print(run_summary_markdown)


In [ ]:
handover_summary = {
    "run_id": RUN_ID,
    "source_count": df_source.count(),
    "valid_count": dq.valid_rows.count(),
    "quarantine_count": dq.quarantine_rows.count(),
    "target_table": TARGET_TABLE,
    "quarantine_table": f"{TARGET_TABLE}_QUARANTINE",
    "dq_rule_count": len(dq.rules),
    "required_source_columns": REQUIRED_SOURCE_COLUMNS,
}
display(handover_summary)

